# NumPy — zaawansowane

**Programowanie w Pythonie II** | Wykład 4
**Politechnika Opolska** | Analityka danych w biznesie

---

## Co dzisiaj?

Zeszły tydzień — podstawy NumPy: tworzenie tablic, indeksowanie, operacje wektorowe, filtrowanie, axis. Dzisiaj wchodzimy głębiej:

```mermaid
graph TD
    A["W03: NumPy podstawy"] --> B["Tworzenie tablic"]
    A --> C["Indeksowanie i slicing"]
    A --> D["Operacje wektorowe"]
    A --> E["axis=0 / axis=1"]
    F["W04: NumPy zaawansowane"] --> G["Broadcasting"]
    F --> H["Reshape i stacking"]
    F --> I["where, select, sort, unique"]
    F --> J["Generowanie danych"]
    F --> L["NaN, digitize, diff"]
    A -.-> F
    F -.-> K["W05: Pandas"]
```

### Dlaczego to ważne?

Broadcasting pozwala operować na tablicach **różnych kształtów** — bez pętli. Reshape zmienia układ danych pod wymagania analizy. Sortowanie, ranking, unikalne wartości — to codzienne narzędzia analityka. Generowanie danych syntetycznych — klucz do testowania i symulacji.

**Po dzisiejszym wykładzie kończymy z czystym NumPy.** Od W05 — Pandas. Ale wszystko z NumPy będzie się przydawać — bo Pandas pod spodem to NumPy z etykietami.

### Po tym wykładzie potrafisz:

1. **Wyjaśnić** mechanizm broadcastingu i stosować go w obliczeniach (Bloom 2-3)
2. **Zmieniać** kształt tablic za pomocą reshape, flatten, stacking (Bloom 3)
3. **Stosować** zaawansowane operacje: where, clip, sort, argsort, unique, korelacja (Bloom 3)
4. **Generować** dane syntetyczne z rozkładów (normal, uniform, poisson) i obliczać statystyki opisowe (Bloom 3)
5. **Obsługiwać** brakujące dane (np.nan) i stosować wielowarunkowe operacje (np.select, np.digitize) (Bloom 3)

In [ ]:
import numpy as np
print(f"NumPy wersja: {np.__version__}")

---

## 1. Broadcasting — operacje na tablicach różnych kształtów

Broadcasting to mechanizm, dzięki któremu NumPy potrafi operować na tablicach **różnych kształtów** — bez pisania pętli. Już go używaliście, nie wiedząc o tym.

### Analogia do Excela

| Co robisz w Excelu | Co robi NumPy |
|---------------------|---------------|
| Kolumna cen x komórka z rabatem — kopiujesz formułę w dół | `ceny * 0.9` — skalar rozciąga się na całą tablicę |
| Tabela cen x wiersz rabatów per kwartał — kopiujesz w dół i w prawo | `ceny * (1 - rabat)` — wektor rozciąga się na wiersze |
| Tabela cen x kolumna premii per produkt — kopiujesz formułę | `ceny * premia` — kolumna rozciąga się na kolumny |

In [ ]:
# Najprostszy broadcasting: tablica x skalar
a = np.array([1, 2, 3, 4])
print(f"a = {a}")
print(f"a * 10 = {a * 10}")  # NumPy rozciąga 10 na [10, 10, 10, 10]

In [ ]:
# Macierz cen: 3 produkty x 4 kwartały
ceny = np.array([[100, 110, 120, 130],
                  [200, 210, 220, 230],
                  [50,  55,  60,  65]])

rabat = np.array([0.05, 0.10, 0.15, 0.20])
print(f"Ceny (3 produkty x 4 kwartały):\n{ceny}")
print(f"Rabat per kwartał: {rabat}")

In [ ]:
# Broadcasting: macierz 3x4 * wektor 1x4
ceny_po_rabacie = ceny * (1 - rabat)
print(f"Ceny po rabacie:\n{ceny_po_rabacie}")
# Każdy kwartał ma inny rabat — działa na WSZYSTKIE produkty naraz

In [ ]:
# Broadcasting kolumnowy — premia per produkt
premia = np.array([[1.2],    # produkt A: +20%
                    [1.0],    # produkt B: bez premii
                    [1.5]])   # produkt C: +50%

ceny_z_premia = ceny * premia
print(f"Premia per produkt: {premia.flatten()}")
print(f"Ceny z premią:\n{ceny_z_premia}")
# Wektor kolumnowy 3x1 x macierz 3x4 — działa na WSZYSTKIE kwartały

### Zasady broadcastingu

NumPy porównuje kształty **od prawej** do lewej. Wymiar musi być **taki sam** lub **równy 1**.

| Kształty | Wynik | Dlaczego |
|----------|-------|----------|
| `(3, 4)` x `(4,)` | `(3, 4)` | 4 == 4 |
| `(3, 4)` x `(3, 1)` | `(3, 4)` | 3 == 3, 1 rozciąga się do 4 |
| `(3, 4)` x `(1, 4)` | `(3, 4)` | 1 rozciąga się do 3, 4 == 4 |
| `(3, 4)` x `(3,)` | **BŁĄD!** | 4 != 3 — nie pasuje |

**Najczęstszy błąd:** macierz 3x4 razy wektor 3 elementów. Rozwiązanie: `.reshape(3, 1)` albo `[:, np.newaxis]`.

In [ ]:
# Błąd broadcastingu — 3x4 razy wektor 3
try:
    blad = ceny * np.array([1, 2, 3])    # 3x4 * 3 — nie pasuje!
except ValueError as e:
    print(f"Błąd: {e}")

In [ ]:
# Rozwiązanie: reshape na kolumnę
mnoznik = np.array([1, 2, 3]).reshape(3, 1)  # (3,) -> (3, 1)
print(f"Mnożnik po reshape: {mnoznik.flatten()}, kształt: {mnoznik.shape}")
print(f"Wynik:\n{ceny * mnoznik}")  # teraz działa: (3,4) x (3,1)

---

## 2. Reshape i stacking — zmiana kształtu tablic

Dane nie zawsze przychodzą w formacie, który potrzebujemy. Reshape i stacking pozwalają reorganizować dane bez ich kopiowania.

### Zestawienie metod

| Metoda | Co robi | Kiedy używać | Uwagi |
|--------|---------|-------------|-------|
| `a.reshape(w, k)` | Zmienia kształt | Dane płaskie -> macierz | Liczba elementów musi się zgadzać |
| `a.reshape(w, -1)` | Auto-wymiar | Nie chcesz liczyć | NumPy oblicza brakujący wymiar |
| `a.flatten()` | Spłaszcza do 1D | Macierz -> wektor | Zwraca **kopię** (bezpieczne) |
| `a.ravel()` | Spłaszcza do 1D | Macierz -> wektor | Zwraca **widok** (szybsze) |
| `np.vstack([a, b])` | Pionowo (wiersze) | Nowe dane pod spodem | Tablice muszą mieć tę samą liczbę kolumn |
| `np.hstack([a, b])` | Poziomo (obok) | Dłuższy wektor | Tablice muszą mieć tę samą liczbę wierszy |
| `np.column_stack([a, b])` | Kolumny obok siebie | Wektory -> macierz | Każdy wektor staje się kolumną |

In [ ]:
# Reshape — zmiana kształtu
a = np.arange(12)
print(f"Flat (12 elementów): {a}")
print(f"Reshape 3x4:\n{a.reshape(3, 4)}")

In [ ]:
# Reshape z -1 — NumPy sam oblicza brakujący wymiar
print(f"Reshape (3, -1) -> 3x{12//3}:\n{a.reshape(3, -1)}")
print(f"Reshape (-1, 6) -> {12//6}x6:\n{a.reshape(-1, 6)}")
# 12 / 3 = 4 kolumny,  12 / 6 = 2 wiersze

In [ ]:
# Reshape 4x3 — inna kolejność niż 3x4
print(f"Reshape 4x3:\n{a.reshape(4, 3)}")
# Uwaga: reshape NIE transponuje! Elementy wypełniają wierszami

In [ ]:
# Flatten vs ravel — spłaszczanie do 1D
macierz = np.array([[1, 2, 3],
                     [4, 5, 6]])
print(f"Macierz:\n{macierz}")
print(f"Flatten: {macierz.flatten()}")    # kopia — bezpieczne
print(f"Ravel:   {macierz.ravel()}")      # widok — szybsze

In [ ]:
# Łączenie tablic — vstack (pionowo)
q1 = np.array([100, 200, 150])
q2 = np.array([120, 210, 160])
q3 = np.array([130, 230, 180])

print(f"vstack (wiersze pod sobą):\n{np.vstack([q1, q2, q3])}")

In [ ]:
# hstack — poziomo (dłuższy wektor)
print(f"hstack (obok siebie): {np.hstack([q1, q2, q3])}")

In [ ]:
# column_stack — każda tablica jako kolumna
tabela = np.column_stack([q1, q2, q3])
print(f"column_stack (kolumny obok siebie):\n{tabela}")
# Przydatne gdy łączysz dane z różnych źródeł — np. Q1, Q2, Q3 jako kolumny

---

## 3. Operacje zaawansowane

### np.where() — warunkowe przypisanie

`np.where(warunek, wartość_True, wartość_False)` — jak `=JEŻELI()` w Excelu, ale na **całej tablicy naraz**.

| Excel | NumPy | Opis |
|-------|-------|------|
| `=JEŻELI(A1>=3; "ZDAŁ"; "NIE ZDAŁ")` | `np.where(oceny >= 3, 'ZDAŁ', 'NIE ZDAŁ')` | Warunek na tablicy |
| `=JEŻELI(A1>200; A1*0.85; A1*0.95)` | `np.where(ceny > 200, ceny*0.85, ceny*0.95)` | Obliczenia warunkowe |

In [ ]:
# np.where — zdał / nie zdał
oceny = np.array([3.0, 4.5, 2.0, 5.0, 3.5, 4.0, 2.5])
status = np.where(oceny >= 3.0, 'ZDAŁ', 'NIE ZDAŁ')
print(f"Oceny:  {oceny}")
print(f"Status: {status}")

In [ ]:
# np.where z imionami — pełny raport
imiona = ['Anna', 'Bartek', 'Celina', 'Dawid', 'Ewa', 'Filip', 'Grażyna']
print("Wyniki egzaminu:")
for imie, ocena, s in zip(imiona, oceny, status):
    print(f"  {imie:8s}: {ocena} -> {s}")

In [ ]:
# np.where z obliczeniami — różne stawki rabatu
ceny_prod = np.array([50, 150, 300, 80, 500, 1200])
ceny_po = np.where(ceny_prod > 200, ceny_prod * 0.85, ceny_prod * 0.95)
print(f"Ceny oryginalne: {ceny_prod}")
print(f"Po rabacie:      {ceny_po}")
# Droższe niż 200 -> rabat 15%, tańsze -> rabat 5%

In [ ]:
# np.where — premia za wyniki
premia = np.where(oceny >= 4.0, 500, 0)
print(f"Oceny:  {oceny}")
print(f"Premia: {premia}")

### np.select() — wielowarunkowe przypisanie

`np.where` obsługuje 2 warianty (tak/nie). A co gdy masz 3, 4, 5 kategorii? Zamiast zagnieżdżać `np.where` w `np.where` — użyj `np.select`.

| Excel | NumPy | Opis |
|-------|-------|------|
| `=IFS(A1>=90;"A"; A1>=70;"B"; ...)` | `np.select(warunki, wartości, default)` | Wiele warunków |
| Zagnieżdżone JEŻELI (3+ poziomy) | `np.select([w1, w2, w3], [v1, v2, v3])` | Czytelniejsze |

In [ ]:
# np.select — segmentacja klientów wg obrotu
obroty = np.array([15000, 3500, 800, 52000, 9200, 1100, 28000])

warunki = [
    obroty >= 20000,     # premium
    obroty >= 5000,      # standard
    obroty >= 1000,      # basic
]
wartosci = ['PREMIUM', 'STANDARD', 'BASIC']

segment = np.select(warunki, wartosci, default='MICRO')
print(f"Obroty:   {obroty}")
print(f"Segmenty: {segment}")

In [ ]:
# np.select — system ocen (jak w Excelu zagnieżdżone IF)
punkty = np.array([95, 72, 58, 83, 44, 91, 67])

ocena = np.select(
    [punkty >= 90, punkty >= 75, punkty >= 60, punkty >= 50],
    ['bardzo dobry', 'dobry', 'dostateczny', 'mierny'],
    default='niedostateczny'
)
print("System ocen:")
for p, o in zip(punkty, ocena):
    print(f"  {p} pkt -> {o}")

### np.clip() — przycinanie wartości do zakresu

`np.clip(tablica, min, max)` — wartości poniżej `min` stają się `min`, powyżej `max` stają się `max`. Przydatne do czyszczenia danych.

In [ ]:
# np.clip — przycinanie wartości do zakresu
oceny_brudne = np.array([1.5, 3.0, 5.5, 4.0, 0.0, 6.0])
oceny_czyste = np.clip(oceny_brudne, 2.0, 5.0)
print(f"Oceny brudne:  {oceny_brudne}")
print(f"Po clip(2, 5): {oceny_czyste}")
# Wartości poza skalą 2-5 przycinane do granic

### Sortowanie i ranking

In [ ]:
# np.sort — posortowana kopia (oryginał bez zmian)
sprzedaz = np.array([340, 120, 560, 90, 420])
print(f"Oryginał:   {sprzedaz}")
print(f"Rosnąco:    {np.sort(sprzedaz)}")
print(f"Malejąco:   {np.sort(sprzedaz)[::-1]}")
print(f"Oryginał:   {sprzedaz}")  # nie zmieniony!

In [ ]:
# np.argsort — indeksy sortowania (do rankingów!)
produkty = ['A', 'B', 'C', 'D', 'E']
indeksy = np.argsort(sprzedaz)[::-1]  # indeksy od najlepszego

print("Ranking sprzedaży (TOP-3):")
for i, idx in enumerate(indeksy[:3]):
    print(f"  {i+1}. {produkty[idx]} — {sprzedaz[idx]} szt.")

In [ ]:
# Ranking studentów po ocenach
oceny_stud = np.array([3.0, 5.0, 4.5, 2.0, 4.0, 3.5])
studenci = ['Anna', 'Bartek', 'Celina', 'Dawid', 'Ewa', 'Filip']
kolejnosc = np.argsort(oceny_stud)[::-1]

print("Ranking ocen:")
for miejsce, idx in enumerate(kolejnosc, 1):
    print(f"  {miejsce}. {studenci[idx]} — {oceny_stud[idx]}")

### np.unique() — unikalne wartości i zliczanie

In [ ]:
# np.unique — unikalne wartości
transakcje = np.array(['IT', 'HR', 'IT', 'Finanse', 'HR', 'IT', 'Finanse', 'Finanse'])
unikalne, ile = np.unique(transakcje, return_counts=True)
print(f"Działy:             {unikalne}")
print(f"Liczba transakcji:  {ile}")
# Jak value_counts() w Pandas — poznacie za tydzień

In [ ]:
# np.unique na liczbach — zliczanie ocen
oceny_grupy = np.array([3, 5, 3, 4, 5, 5, 3, 4, 2, 4])
wartosci, liczby = np.unique(oceny_grupy, return_counts=True)
print("Rozkład ocen:")
for w, l in zip(wartosci, liczby):
    print(f"  Ocena {w}: {l} student(ów)")

### np.digitize() — przypisywanie do przedziałów (binning)

`np.digitize(dane, bins)` — przypisuje każdy element do przedziału. Jak tworzenie grup wiekowych, przedziałów cenowych, klas dochodowych. Przygotowanie do `pd.cut()` z Pandas (W05).

In [ ]:
# np.digitize — grupy wiekowe klientów
wiek = np.array([22, 35, 48, 19, 67, 55, 31, 42, 28, 73])
bins_wiek = [18, 30, 45, 60]  # granice przedziałów

grupy = np.digitize(wiek, bins_wiek)
etykiety = ['<18', '18-30', '31-45', '46-60', '60+']

print("Segmentacja wiekowa:")
for w, g in zip(wiek, grupy):
    print(f"  Wiek {w} -> grupa {etykiety[g]}")

# Ile klientów w każdej grupie?
for i, et in enumerate(etykiety):
    print(f"  {et}: {(grupy == i).sum()} os.")

In [ ]:
# np.digitize — przedziały cenowe
ceny_produktow = np.array([29, 150, 499, 75, 1200, 350, 15, 89])
bins_ceny = [50, 100, 500]

kategorie_idx = np.digitize(ceny_produktow, bins_ceny)
kategorie = ['budżetowy', 'ekonomiczny', 'premium', 'luksusowy']

print("Kategorie cenowe:")
for c, ki in zip(ceny_produktow, kategorie_idx):
    print(f"  {c} zł -> {kategorie[ki]}")

### Korelacja — np.corrcoef()

In [ ]:
# Korelacja: reklama vs sprzedaż
np.random.seed(42)
reklama = np.random.normal(1000, 200, 30)     # budżet reklamy
sprzedaz_kor = 500 + 0.8 * reklama + np.random.normal(0, 100, 30)

r = np.corrcoef(reklama, sprzedaz_kor)[0, 1]
print(f"Korelacja reklama vs sprzedaż: r = {r:.3f}")
# r ~ 0.85 — silna dodatnia korelacja

In [ ]:
# Macierz korelacji — pełna
korelacja = np.corrcoef(reklama, sprzedaz_kor)
print(f"Macierz korelacji:\n{np.round(korelacja, 3)}")
# [0,0] i [1,1] = 1.0 (korelacja z samym sobą)
# [0,1] i [1,0] = r (korelacja między zmiennymi)

**Uwaga:** Korelacja to **nie** przyczynowość! `r = 0.85` oznacza, że zmienne zmieniają się razem, ale nie mówi, która powoduje drugą. Szerzej o tym — na W11 (statystyka).

---

## 6. Generowanie danych i statystyki opisowe

W pracy analityka często potrzebujesz danych do testów albo symulacji. NumPy oferuje kilka rozkładów:

| Rozkład | Funkcja | Parametry | Przykład zastosowania |
|---------|---------|-----------|----------------------|
| Normalny | `np.random.normal(loc, scale, size)` | średnia, odchylenie | Wynagrodzenia, wzrost, pomiary |
| Jednostajny | `np.random.uniform(low, high, size)` | min, max | Ceny, losowe wartości w przedziale |
| Poissona | `np.random.poisson(lam, size)` | średnia liczba zdarzeń | Liczba zamówień, awarii, klientów |
| Całkowite | `np.random.randint(low, high, size)` | min, max (bez max) | Oceny, ilości, ID |

**`np.random.seed(42)`** — ustawia ziarno generatora. Te same dane za każdym razem. Kluczowe w analizie — wyniki muszą być **powtarzalne**.

---

## 5. Brakujące dane — np.nan

W realnych danych **zawsze** są braki. Klient nie podał wieku, czujnik się zepsuł, ankieta nie wypełniona do końca. NumPy reprezentuje brak danych jako `np.nan` (Not a Number).

| Problem | Zwykłe funkcje | Funkcje nan-safe |
|---------|---------------|------------------|
| Średnia z brakami | `a.mean()` -> `nan` | `np.nanmean(a)` -> poprawna średnia |
| Suma z brakami | `a.sum()` -> `nan` | `np.nansum(a)` -> poprawna suma |
| Std z brakami | `a.std()` -> `nan` | `np.nanstd(a)` -> poprawne std |

**Zasada:** Jeden `nan` "zaraża" cały wynik. Dlatego istnieją funkcje `nan*` — ignorują braki.

In [ ]:
# np.nan — brakujące dane w tablicy
temperatura = np.array([21.5, 22.0, np.nan, 23.1, np.nan, 20.8, 22.5])
print(f"Temperatura: {temperatura}")
print(f"Zwykła średnia:  {temperatura.mean()}")     # nan!
print(f"nanmean:         {np.nanmean(temperatura):.1f}")  # ignoruje braki
print(f"nansum:          {np.nansum(temperatura):.1f}")
print(f"nanstd:          {np.nanstd(temperatura):.1f}")

In [ ]:
# Ile brakuje? Gdzie brakuje?
print(f"Ile NaN:         {np.isnan(temperatura).sum()}")
print(f"Gdzie NaN:       {np.where(np.isnan(temperatura))[0]}")
print(f"Ile poprawnych:  {np.count_nonzero(~np.isnan(temperatura))}")
# ~ (tylda) odwraca True/False

In [ ]:
# Realistyczny przykład — wynagrodzenia z brakami
wynagrodzenia_nan = np.array([4500, np.nan, 5200, 6100, np.nan, 3800, 5500, np.nan, 4900])
print(f"Dane: {wynagrodzenia_nan}")
print(f"Braki: {np.isnan(wynagrodzenia_nan).sum()} z {len(wynagrodzenia_nan)}")
print(f"Średnia (bez braków): {np.nanmean(wynagrodzenia_nan):.0f} zł")
print(f"Mediana (bez braków): {np.nanmedian(wynagrodzenia_nan):.0f} zł")
print(f"Min: {np.nanmin(wynagrodzenia_nan):.0f}, Max: {np.nanmax(wynagrodzenia_nan):.0f}")
# Na W05 poznacie Pandas — tam obsługa NaN jest jeszcze wygodniejsza

In [ ]:
# Rozkład normalny — wynagrodzenia (średnia 5000, odchylenie 1000)
np.random.seed(42)
wynagrodzenia = np.random.normal(loc=5000, scale=1000, size=100)
print(f"Wynagrodzenia (pierwszych 10): {wynagrodzenia[:10].astype(int)}")
print(f"Średnia: {wynagrodzenia.mean():.0f}, Std: {wynagrodzenia.std():.0f}")

In [ ]:
# Rozkład jednostajny — ceny w przedziale [10, 500]
ceny_losowe = np.random.uniform(low=10, high=500, size=50)
print(f"Ceny (pierwszych 8): {np.round(ceny_losowe[:8], 2)}")
print(f"Min: {ceny_losowe.min():.0f}, Max: {ceny_losowe.max():.0f}")
# Każda wartość z przedziału [10, 500] równie prawdopodobna

In [ ]:
# Rozkład Poissona — liczba zamówień dziennie (średnio 20)
zamowienia = np.random.poisson(lam=20, size=30)
print(f"Zamówienia dziennie (30 dni): {zamowienia}")
print(f"Średnia: {zamowienia.mean():.1f}, Min: {zamowienia.min()}, Max: {zamowienia.max()}")

In [ ]:
# Losowy wybór z istniejącej tablicy
kolory = np.array(['czerwony', 'zielony', 'niebieski'])
losowe_kolory = np.random.choice(kolory, size=8)
print(f"Losowe kolory: {losowe_kolory}")

### Statystyki opisowe — percentyle, IQR, mediana

| Statystyka | Funkcja NumPy | Excel | Opis |
|------------|--------------|-------|------|
| Średnia | `a.mean()` | `=ŚREDNIA()` | Arytmetyczna |
| Mediana | `np.median(a)` | `=MEDIANA()` | Wartość środkowa |
| Odchylenie std | `a.std()` | `=ODCH.STAND()` | Rozproszenie |
| Wariancja | `a.var()` | `=WARIANCJA()` | std do kwadratu |
| Percentyl | `np.percentile(a, q)` | `=PERCENTYL()` | q% danych poniżej |
| Suma kumulacyjna | `a.cumsum()` | — | Narastająco |
| Iloczyn kumulacyjny | `a.cumprod()` | — | Narastająco |
| Iloczyn wszystkich | `a.prod()` | `=ILOCZYN()` | Wszystkie elementy |

In [ ]:
# Pełny zestaw statystyk opisowych
dane = wynagrodzenia
print(f"Średnia:  {dane.mean():.0f}")
print(f"Mediana:  {np.median(dane):.0f}")
print(f"Std:      {dane.std():.0f}")
print(f"Min:      {dane.min():.0f}")
print(f"Max:      {dane.max():.0f}")

In [ ]:
# Percentyle i IQR
q1 = np.percentile(dane, 25)
q3 = np.percentile(dane, 75)
iqr = q3 - q1
print(f"Q1 (25%): {q1:.0f}")
print(f"Q3 (75%): {q3:.0f}")
print(f"IQR:      {iqr:.0f}")
# IQR = rozstęp międzykwartylowy — miara rozproszenia odporna na outlier'y

In [ ]:
# Suma kumulacyjna — narastające przychody
miesieczne = np.array([100, 150, 120, 200, 180, 160])
print(f"Przychody miesięczne: {miesieczne}")
print(f"Suma kumulacyjna:     {miesieczne.cumsum()}")
# [100, 250, 370, 570, 750, 910] — ile łącznie po każdym miesiącu

### np.diff() — zmiany między kolejnymi elementami

`np.diff(a)` — oblicza różnicę między kolejnymi elementami. Kluczowe w analizie dynamiki: zmiana miesiąc do miesiąca, dzień do dnia, kwartał do kwartału.

In [ ]:
# np.diff — dynamika sprzedaży miesiąc do miesiąca
sprzedaz_mies = np.array([100, 120, 115, 140, 135, 160])
zmiany = np.diff(sprzedaz_mies)
print(f"Sprzedaż:          {sprzedaz_mies}")
print(f"Zmiany m/m:        {zmiany}")
print(f"Wzrosty:           {(zmiany > 0).sum()} miesięcy")
print(f"Spadki:            {(zmiany < 0).sum()} miesięcy")
print(f"Największy wzrost: +{zmiany.max()} (miesiąc {zmiany.argmax() + 2})")

In [ ]:
# np.diff — zmiana procentowa (jak w Excelu)
zmiana_proc = np.diff(sprzedaz_mies) / sprzedaz_mies[:-1] * 100
print("Zmiany procentowe m/m:")
for i, z in enumerate(zmiana_proc):
    symbol = '+' if z > 0 else ''
    print(f"  Miesiąc {i+1} -> {i+2}: {symbol}{z:.1f}%")

---

## 7. Kompletna ściąga NumPy zaawansowane — jak to zrobić?

### Zmiana kształtu i łączenie

| Chcę... | Kod | Uwagi |
|---------|-----|-------|
| Zmienić kształt | `a.reshape(3, 4)` | Liczba elementów musi się zgadzać |
| Auto-wymiar | `a.reshape(3, -1)` | NumPy oblicza brakujący wymiar |
| Spłaszczyć do 1D | `a.flatten()` | Zwraca kopię (bezpieczne) |
| Spłaszczyć szybko | `a.ravel()` | Zwraca widok (zmiana wpływa na oryginał) |
| Połączyć tablice | `np.concatenate([a, b])` | Wzdłuż istniejącej osi |
| Złożyć pionowo | `np.vstack([a, b])` | Wiersze pod sobą |
| Złożyć poziomo | `np.hstack([a, b])` | Obok siebie |
| Wektory jako kolumny | `np.column_stack([a, b])` | Każdy wektor = kolumna |

### Operacje warunkowe

| Chcę... | Kod | Uwagi |
|---------|-----|-------|
| Warunkowe przypisanie | `np.where(warunek, tak, nie)` | Jak JEŻELI() w Excelu |
| Wielowarunkowe (N kategorii) | `np.select([w1, w2], [v1, v2], default)` | Jak zagnieżdżone JEŻELI() |
| Przyciąć do zakresu | `np.clip(a, min, max)` | Wartości poza zakresem -> granice |

### Sortowanie i unikalne

| Chcę... | Kod | Uwagi |
|---------|-----|-------|
| Posortować (kopia) | `np.sort(a)` | Oryginał bez zmian |
| Posortować malejąco | `np.sort(a)[::-1]` | Sortuj + odwróć |
| Indeksy sortowania | `np.argsort(a)` | Do rankingów |
| Unikalne wartości | `np.unique(a)` | Posortowane |
| Unikalne + ile razy | `np.unique(a, return_counts=True)` | Zwraca 2 tablice |
| Przypisać do przedziałów | `np.digitize(a, bins)` | Binning, grupy wiekowe |
| Korelację | `np.corrcoef(x, y)[0, 1]` | Pearsona, zakres [-1, 1] |

### Generowanie danych

| Chcę... | Kod | Uwagi |
|---------|-----|-------|
| Rozkład normalny | `np.random.normal(loc, scale, size)` | Parametry: średnia, std |
| Rozkład jednostajny | `np.random.uniform(low, high, size)` | Równomierne z przedziału |
| Rozkład Poissona | `np.random.poisson(lam, size)` | Zliczenia zdarzeń |
| Losowe z tablicy | `np.random.choice(tablica, size)` | Losowy wybór |
| Powtarzalność | `np.random.seed(42)` | Przed losowaniem |

### Statystyki rozszerzone

| Chcę... | Kod | Excel |
|---------|-----|-------|
| Medianę | `np.median(a)` | `=MEDIANA()` |
| Percentyl | `np.percentile(a, 25)` | `=PERCENTYL()` |
| Wariancję | `a.var()` | `=WARIANCJA()` |
| Sumę kumulacyjną | `a.cumsum()` | — |
| Iloczyn kumulacyjny | `a.cumprod()` | — |
| Różnice kolejnych | `np.diff(a)` | — |
| Zmianę procentową | `np.diff(a) / a[:-1] * 100` | — |
| Średnią (z NaN) | `np.nanmean(a)` | — |
| Sumę (z NaN) | `np.nansum(a)` | — |
| Medianę (z NaN) | `np.nanmedian(a)` | — |
| Znaleźć NaN | `np.isnan(a)` | — |
| Zaokrąglić | `np.round(a, 2)` | `=ZAOKR()` |
| Wartość bezwzględną | `np.abs(a)` | `=MODUŁ.LICZBY()` |
| Pierwiastek | `np.sqrt(a)` | `=PIERWIASTEK()` |
| Logarytm naturalny | `np.log(a)` | `=LN()` |
| Logarytm dziesiętny | `np.log10(a)` | `=LOG()` |

---

## 8. Źródła i dalsze materiały

| Źródło | Opis |
|--------|------|
| **NumPy Broadcasting** | numpy.org/doc/stable/user/basics.broadcasting.html |
| **NumPy Array Manipulation** | numpy.org/doc/stable/reference/routines.array-manipulation.html |
| **NumPy Sorting** | numpy.org/doc/stable/reference/routines.sort.html |
| **NumPy Random** | numpy.org/doc/stable/reference/random/ |
| **NumPy Statistics** | numpy.org/doc/stable/reference/routines.statistics.html |
| **Real Python — Broadcasting** | realpython.com/numpy-array-programming/ |

**Wskazówka:** W Jupyter Notebook — `np.broadcast_shapes?` pokaże dokumentację broadcastingu.

---

## 9. Mini-ćwiczenia na wykładzie

Spróbuj odpowiedzieć na pytania **zanim** zobaczysz rozwiązanie.

### Ćwiczenie A — Broadcasting w sklepie

In [ ]:
# Dane: 4 produkty x 3 miesiące, ceny w złotych
ceny_sklep = np.array([[25, 28, 30],
                        [100, 95, 110],
                        [45, 50, 48],
                        [200, 210, 195]])
produkty_sklep = ['Kubek', 'Plecak', 'Koszulka', 'Buty']
miesiace = ['Styczeń', 'Luty', 'Marzec']
print(f"Ceny (4 produkty x 3 miesiące):\n{ceny_sklep}")

In [ ]:
# 1. Podwyżka 10% na wszystko (broadcasting skalarny)
print(f"Po podwyżce 10%:\n{np.round(ceny_sklep * 1.10, 2)}")

In [ ]:
# 2. Rabat sezonowy: Sty=0%, Lut=5%, Mar=15% (broadcasting wierszowy)
rabat_sezon = np.array([0.0, 0.05, 0.15])
print(f"Rabat sezonowy: {rabat_sezon}")
print(f"Ceny po rabacie:\n{np.round(ceny_sklep * (1 - rabat_sezon), 2)}")

In [ ]:
# 3. Marża per produkt: Kubek 50%, Plecak 30%, Koszulka 40%, Buty 25%
marza = np.array([[1.50], [1.30], [1.40], [1.25]])  # kolumnowy!
print(f"Ceny z marżą:\n{np.round(ceny_sklep * marza, 2)}")

### Ćwiczenie B — Reshape i analiza kwartalnych wyników

In [ ]:
# Dane płaskie — 12 miesięcy sprzedaży (szt.)
sprzedaz_flat = np.array([120, 135, 110, 145, 160, 155, 170, 180, 165, 190, 200, 210])
print(f"Sprzedaż (12 miesięcy): {sprzedaz_flat}")

In [ ]:
# 1. Reshape na kwartały (4 kwartały x 3 miesiące)
kwartaly_mat = sprzedaz_flat.reshape(4, 3)
print(f"Kwartały (4x3):\n{kwartaly_mat}")

In [ ]:
# 2. Suma per kwartał (axis=1)
sumy_kwartal = kwartaly_mat.sum(axis=1)
for i, s in enumerate(sumy_kwartal, 1):
    print(f"  Q{i}: {s} szt.")

In [ ]:
# 3. Najlepszy kwartał i spłaszczenie z powrotem
print(f"Najlepszy kwartał: Q{sumy_kwartal.argmax() + 1} ({sumy_kwartal.max()} szt.)")
print(f"Z powrotem flat: {kwartaly_mat.flatten()}")

### Ćwiczenie C — Analiza danych finansowych (kompleksowe)

In [ ]:
# Firma z 5 oddziałami — przychody miesięczne (tys. zł)
np.random.seed(123)
oddzialy = ['Warszawa', 'Kraków', 'Wrocław', 'Gdańsk', 'Poznań']
przychody = np.random.randint(50, 200, size=(5, 12))  # 5 oddziałów x 12 miesięcy
print(f"Kształt: {przychody.shape}")

In [ ]:
# 1. Który oddział miał najwyższy roczny przychód?
roczne = przychody.sum(axis=1)
print(f"Najwyższy roczny: {oddzialy[roczne.argmax()]} ({roczne.max()} tys. zł)")

In [ ]:
# 2. Który miesiąc był najlepszy łącznie?
miesieczne_suma = przychody.sum(axis=0)
print(f"Najlepszy miesiąc: {miesieczne_suma.argmax() + 1} ({miesieczne_suma.max()} tys. zł)")

In [ ]:
# 3. Ile oddziałów miało średni miesięczny przychód powyżej 100?
srednie = przychody.mean(axis=1)
print(f"Średnie per oddział: {np.round(srednie, 1)}")
print(f"Oddziały > 100 śr.: {(srednie > 100).sum()}")

In [ ]:
# 4. Korelacja Warszawa-Kraków
r = np.corrcoef(przychody[0], przychody[1])[0, 1]
print(f"Korelacja Warszawa-Kraków: r = {r:.3f}")

In [ ]:
# 5. Percentyle i IQR dla Warszawy
waw = przychody[0]
q1_waw = np.percentile(waw, 25)
q3_waw = np.percentile(waw, 75)
print(f"Warszawa — Q1: {q1_waw:.0f}, Mediana: {np.median(waw):.0f}, Q3: {q3_waw:.0f}")
print(f"IQR: {q3_waw - q1_waw:.0f} tys. zł")

---

## Podgląd laboratorium

Na laboratorium przećwiczycie to wszystko na danych biznesowych. Oto co was czeka:

**Start — 3 komendy:**
```
cd C:\Users\student\python2
.venv\Scripts\Activate.ps1
code .
```

**Ćwiczenie 1:** Broadcasting w sklepie — ceny 4 produktów w 3 sklepach, inflacja, rabaty per sklep, marża per produkt

**Ćwiczenie 2:** Reshape — dane płaskie z 12 miesięcy -> macierz kwartałów, łączenie danych z oddziałów (vstack)

**Ćwiczenie 3:** Samodzielna analiza finansowa — wynagrodzenia (mediana, IQR), system premii (zagnieżdżony np.where), ranking pracowników (argsort), korelacja staż-pensja, broadcasting cen produktów

**Ćwiczenie 4:** Generowanie 100 klientów — rozkład normalny (wiek), Poisson (wizyty), segmentacja np.where

**Ćwiczenie 5:** Commit na GitHub

---

## Na koniec

> *"Don't wish it were easier, wish you were better."*
> — Jim Rohn, *The Treasury of Quotes*

**Ciekawostka:** Operacje wektorowe NumPy wykorzystują instrukcje **SIMD** procesora (Single Instruction, Multiple Data) — ten sam mechanizm, który przyspiesza grafikę w grach komputerowych. Broadcasting to jeden z powodów, dla których NumPy operuje na danych **10-100x szybciej** niż pętla w czystym Pythonie.

**Za tydzień:** Pandas — DataFrame, wczytywanie CSV, selekcja danych. NumPy robił obliczenia — Pandas doda etykiety, nazwy kolumn i mnóstwo wygodnych metod.

**Zadanie domowe (nieoceniane):** Wygenerujcie dane o 50 pracownikach: pensja (rozkład normalny, średnia 6000, std 1500), staż (randint 1-20), ocena roczna (uniform 1-5). Policzcie korelację między stażem a pensją. Wrzućcie notebook na GitHub.